<a href="https://colab.research.google.com/github/dipti-2211/Cloud9/blob/main/Cloud9ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install rasterio geopandas rasterstats shapely fiona osmnx xarray netCDF4 -q


In [ ]:
import osmnx as ox
import geopandas as gpd

DISTRICT_NAME = "Dima Hasao, Assam, India"
boundary = ox.geocode_to_gdf(DISTRICT_NAME)
boundary.to_file("boundary.geojson", driver="GeoJSON")

print("Boundary CRS:", boundary.crs)
print("Boundary bounds:", boundary.total_bounds)

Boundary CRS: epsg:4326
Boundary bounds: [92.5205147 24.9710763 93.4734857 25.8289431]


In [ ]:
import rasterio
from rasterio.mask import mask

with rasterio.open("DEM.tif") as src:
    print("DEM CRS:", src.crs)
    print("DEM bounds:", src.bounds)
    clipped, transform = mask(src, boundary.geometry, crop=True)
    meta = src.meta.copy()
    meta.update({"height": clipped.shape[1], "width": clipped.shape[2], "transform": transform})

with rasterio.open("dem_clipped.tif", "w", **meta) as dst:
    dst.write(clipped)

print("Clipped OK — if this ran without a WindowError, DEM and boundary now overlap.")

DEM CRS: EPSG:4326
DEM bounds: BoundingBox(left=92.52041666670272, bottom=24.971249999995216, right=93.4734722222584, top=25.829027777773106)
Clipped OK — if this ran without a WindowError, DEM and boundary now overlap.


In [ ]:
from rasterio.warp import calculate_default_transform, reproject, Resampling

dst_crs = "EPSG:32646"

with rasterio.open("dem_clipped.tif") as src:
    transform, width, height = calculate_default_transform(src.crs, dst_crs, src.width, src.height, *src.bounds)
    meta = src.meta.copy()
    meta.update({"crs": dst_crs, "transform": transform, "width": width, "height": height})
    with rasterio.open("dem_reprojected.tif", "w", **meta) as dst:
        reproject(
            source=rasterio.band(src, 1), destination=rasterio.band(dst, 1),
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=transform, dst_crs=dst_crs, resampling=Resampling.bilinear
        )

print("Reprojected to UTM 46N.")

Reprojected to UTM 46N.


In [ ]:
import numpy as np

with rasterio.open("dem_reprojected.tif") as src:
    dem_arr = src.read(1).astype(float)
    dem_transform = src.transform
    dem_meta = src.meta.copy()
    nodata = src.nodata

if nodata is not None:
    dem_arr[dem_arr == nodata] = np.nan

px_width = dem_transform.a       # meters per pixel, x direction
px_height = -dem_transform.e     # meters per pixel, y direction (positive)

# Gradient: axis 0 = rows (north-south), axis 1 = cols (east-west)
dzdy, dzdx = np.gradient(dem_arr, px_height, px_width)

slope_rad = np.arctan(np.sqrt(dzdx**2 + dzdy**2))
slope_deg = np.degrees(slope_rad)

aspect_rad = np.arctan2(dzdy, -dzdx)
aspect_deg = (90 - np.degrees(aspect_rad)) % 360

def save_raster(path, array, meta):
    meta = meta.copy()
    meta.update(dtype="float32", count=1)
    with rasterio.open(path, "w", **meta) as dst:
        dst.write(array.astype("float32"), 1)

save_raster("slope.tif", slope_deg, dem_meta)
save_raster("aspect.tif", aspect_deg, dem_meta)

print("Slope/aspect computed. Slope range:", np.nanmin(slope_deg), "-", np.nanmax(slope_deg), "degrees")

Slope/aspect computed. Slope range: 0.0 - 73.19007182382053 degrees


In [ ]:
# ============================================================
# REPLACES CELLS 6 AND 7 — merge these into a single cell
# ============================================================
import os
import pdfplumber
import pandas as pd

if os.path.exists("gsi_full_inventory_cache.csv"):
    df = pd.read_csv("gsi_full_inventory_cache.csv")
    print("Loaded from cache:", len(df), "rows — skipped the slow PDF parse")

else:
    col_names = ['Sl_No', 'Slide_No', 'State', 'District', 'Slide_Name', 'NH_SH_Location',
                 'Latitude', 'Longitude', 'Material_Involved', 'Movement_Type', 'History']

    data_rows = []
    with pdfplumber.open("landslide_report_Assam.pdf") as pdf:
        for page in pdf.pages:
            for t in page.extract_tables():
                for row in t:
                    if row[0] in (None, 'Sl.No.') or row[0] is None:
                        continue
                    if len(row) == len(col_names):
                        data_rows.append(row)

    df = pd.DataFrame(data_rows, columns=col_names)
    print("Total rows extracted (fresh parse):", len(df))

    df.to_csv("gsi_full_inventory_cache.csv", index=False)
    print("Cache saved for next run.")

# Filter to Dima Hasao — the PDF uses a few different spellings for this district
# (Dima Hasao, Dima Hasao (N.C. Hills), Dima Hasao (NC Hills)), so match loosely
mask = df['District'].str.strip().str.lower().str.contains('dima hasao|nc hills|n.c. hills', na=False, regex=True)
dima_hasao_df = df[mask].copy()
print("Dima Hasao rows:", len(dima_hasao_df))

# Clean coordinates — some entries have blanks, stray characters, or comments in the fields
dima_hasao_df['Latitude'] = pd.to_numeric(dima_hasao_df['Latitude'], errors='coerce')
dima_hasao_df['Longitude'] = pd.to_numeric(dima_hasao_df['Longitude'], errors='coerce')
dima_hasao_df = dima_hasao_df.dropna(subset=['Latitude', 'Longitude'])
print("Usable rows after cleaning coordinates:", len(dima_hasao_df))

dima_hasao_df.to_csv("dima_hasao_landslides_raw.csv", index=False)

import geopandas as gpd
from shapely.geometry import Point

geometry = [Point(xy) for xy in zip(dima_hasao_df['Longitude'], dima_hasao_df['Latitude'])]
landslides = gpd.GeoDataFrame(dima_hasao_df, geometry=geometry, crs="EPSG:4326")
landslides = landslides.to_crs(dst_crs)
landslides["label"] = 1

print("Final landslide point count for Dima Hasao:", len(landslides))

Loaded from cache: 36072 rows — skipped the slow PDF parse
Dima Hasao rows: 417
Usable rows after cleaning coordinates: 417
Final landslide point count for Dima Hasao: 417


In [ ]:
# Cache the full extracted table so re-running never re-parses the 904-page PDF again
df.to_csv("gsi_full_inventory_cache.csv", index=False)

In [ ]:
!pip install pdfplumber -q

import pdfplumber

with pdfplumber.open("landslide_report_Assam.pdf") as pdf:
    print(f"Total pages: {len(pdf.pages)}")
    for i, page in enumerate(pdf.pages):
        tables = page.extract_tables()
        if tables:
            print(f"\n--- Page {i+1}: found {len(tables)} table(s) ---")
            for t in tables:
                for row in t[:3]:   # just first 3 rows as a preview
                    print(row)

Total pages: 904

--- Page 1: found 1 table(s) ---
['LANDSLIDE INVENTORY (Field vaidated)', None, None, None, None, None, None, None, None, None, None]
['Sl.No.', 'Slide_No', 'State', 'District', 'Slide_Name', 'NH_SH_Location', 'Latitude', 'Longitude', 'Material Involved', 'Movement\nType', 'History']
['1', 'ASM/HKN/83D07/2020/2', 'Assam', 'Hailakandi', 'Kukinala slide', 'Kukinala', '24.27', '92.5', 'Debris', 'Slide', 'NA']

--- Page 2: found 1 table(s) ---
['Sl.No.', 'Slide_No', 'State', 'District', 'Slide_Name', 'NH_SH_Location', 'Latitude', 'Longitude', 'Material Involved', 'Movement\nType', 'History']
['40', 'ASM/HLK/83D09/2020/006', 'Assam', 'Hailakandi', 'Chandipur Grant Slide', 'Bowarthar, Chandipur, Algapur\nCircle.', '24.79489', '92.55506', 'Debris', 'Slide', '02 June 2020']
['41', 'AS/DIM/83C16/2018/27', 'Assam', 'Cachar', 'Lowajuri', 'NH53', '24.795892', '93.045425', 'Debris', 'Slide', 'NA']

--- Page 3: found 1 table(s) ---
['Sl.No.', 'Slide_No', 'State', 'District', 'Slide

In [ ]:
import osmnx as ox

roads_graph = ox.graph_from_polygon(boundary.geometry.iloc[0], network_type="drive")
roads_gdf = ox.graph_to_gdfs(roads_graph, nodes=False)
roads_gdf.to_file("roads.geojson", driver="GeoJSON")

print("Roads extracted:", len(roads_gdf), "segments")


Roads extracted: 3396 segments


In [ ]:
import numpy as np
from shapely.geometry import Point
import geopandas as gpd

boundary_utm = boundary.to_crs(dst_crs)
minx, miny, maxx, maxy = boundary_utm.total_bounds
n = len(landslides)

random_points = []
while len(random_points) < n:
    p = Point(np.random.uniform(minx, maxx), np.random.uniform(miny, maxy))
    if boundary_utm.contains(p).any():
        random_points.append(p)

negatives = gpd.GeoDataFrame(geometry=random_points, crs=dst_crs)
negatives["label"] = 0

print("Negative points generated:", len(negatives))

Negative points generated: 417


In [ ]:
import pandas as pd

points = gpd.GeoDataFrame(
    pd.concat([landslides[["geometry", "label"]], negatives[["geometry", "label"]]], ignore_index=True),
    crs=dst_crs
)

print("Total training points:", len(points))
print("Label counts:", points["label"].value_counts().to_dict())


Total training points: 834
Label counts: {1: 417, 0: 417}


In [ ]:
from rasterstats import point_query

points["elevation"] = point_query(points, "dem_reprojected.tif")
points["slope"] = point_query(points, "slope.tif")
points["aspect"] = point_query(points, "aspect.tif")

print(points[["elevation", "slope", "aspect"]].describe())

         elevation        slope       aspect
count  1407.000000  1401.000000  1401.000000
mean    563.243274    16.177205   175.542125
std     307.367763     8.146694    97.169583
min      83.517195     0.314669     0.820042
25%     339.694114    10.067945    92.561407
50%     515.164199    15.714367   174.076943
75%     707.097164    21.169053   256.637244
max    1716.988510    48.824283   357.209303


In [ ]:
roads = gpd.read_file("roads.geojson").to_crs(dst_crs)
points = points.drop(columns=["index_right"], errors="ignore").reset_index(drop=True)
points = gpd.sjoin_nearest(points, roads, distance_col="dist_to_road")
points = points[~points.index.duplicated(keep="first")]

print("De-duplicated point count:", len(points))
print(points["dist_to_road"].describe())

/usr/local/lib/python3.13/dist-packages/geopandas/io/file.py:576: UserWarning: Could not parse column 'reversed' as JSON; leaving as string
  return pyogrio.read_dataframe(path_or_bytes, bbox=bbox, **kwargs)


De-duplicated point count: 1472
count      1472.000000
mean       2262.232523
std       11597.106895
min           0.194284
25%          51.084763
50%         460.576227
75%        2037.040299
max      183870.345136
Name: dist_to_road, dtype: float64


In [ ]:
import xarray as xr

ds = xr.open_dataset("RF25_ind2024_rfp25.nc")
print(ds)


<xarray.Dataset> Size: 26MB
Dimensions:    (TIME: 366, LATITUDE: 129, LONGITUDE: 135)
Coordinates:
  * TIME       (TIME) datetime64[ns] 3kB 2024-01-01 2024-01-02 ... 2024-12-31
  * LATITUDE   (LATITUDE) float64 1kB 6.5 6.75 7.0 7.25 ... 38.0 38.25 38.5
  * LONGITUDE  (LONGITUDE) float64 1kB 66.5 66.75 67.0 ... 99.5 99.75 100.0
Data variables:
    RAINFALL   (TIME, LATITUDE, LONGITUDE) float32 25MB ...
Attributes:
    history:      FERRET V6.82    9-Apr-26
    Conventions:  CF-1.0


In [ ]:
RAIN_VAR = "RAINFALL"
LAT_NAME = "LATITUDE"
LON_NAME = "LONGITUDE"

time_dim = "TIME" if "TIME" in ds.dims else ("time" if "time" in ds.dims else None)
rain_mean = ds[RAIN_VAR].mean(dim=time_dim) if time_dim else ds[RAIN_VAR]

points_latlon = points.to_crs("EPSG:4326")

rainfall_values = []
for geom in points_latlon.geometry:
    val = rain_mean.sel({LAT_NAME: geom.y, LON_NAME: geom.x}, method="nearest").values.item()
    rainfall_values.append(val)

points["rainfall"] = rainfall_values

print(points["rainfall"].describe())

count    1472.000000
mean        5.384847
std         1.799961
min         2.765881
25%         3.475105
50%         4.997047
75%         6.778077
max         8.737246
Name: rainfall, dtype: float64


In [ ]:
points.drop(columns=["geometry", "index_right"], errors="ignore").to_csv("landslide_points.csv", index=False)

print("Saved landslide_points.csv with", len(points), "rows")
print(points.head())

Saved landslide_points.csv with 1472 rows
                         geometry  label   elevation      slope      aspect  \
0  POINT (516548.805 2765289.944)      1  210.068974  16.374961  261.680795   
1  POINT (516548.805 2765289.944)      1  210.068974  16.374961  261.680795   
2  POINT (504036.127 2765834.172)      1         NaN        NaN         NaN   
3  POINT (504036.127 2765834.172)      1         NaN        NaN         NaN   
4  POINT (475260.987 2766119.379)      1         NaN        NaN         NaN   

        u_left       v_left  key_left  \
0  12205921413   6263645330         0   
1   6263645330  12205921413         0   
2  12205988597  12205991116         0   
3  12205991116  12205988597         0   
4   9784622508   9784757148         0   

                                     osmid_left    highway_left  ...  \
0  [668863304, 667217706, 667217692, 525641655]  [unclassified]  ...   
1  [668863304, 667217706, 667217692, 525641655]  [unclassified]  ...   
2                   

In [ ]:
points.to_csv("landslide_points.csv", index=False)

In [ ]:
from google.colab import files

files.download("landslide_points.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd

df = pd.read_csv("landslide_points.csv")
print("Rows loaded:", len(df))
print("Columns:", list(df.columns))



Rows loaded: 1472
Columns: ['geometry', 'label', 'elevation', 'slope', 'aspect', 'u_left', 'v_left', 'key_left', 'osmid_left', 'highway_left', 'oneway_left', 'reversed_left', 'length_left', 'bridge_left', 'ref_left', 'lanes_left', 'junction_left', 'name_left', 'tunnel_left', 'dist_to_road', 'rainfall', 'index_right', 'u_right', 'v_right', 'key_right', 'osmid_right', 'highway_right', 'oneway_right', 'reversed_right', 'length_right', 'bridge_right', 'ref_right', 'lanes_right', 'junction_right', 'name_right', 'tunnel_right']


In [ ]:
feature_cols = ["elevation", "slope", "aspect", "dist_to_road", "rainfall"]
target_col = "label"

data = df[feature_cols + [target_col]].copy()

before = len(data)
data = data.dropna()
after = len(data)
print(f"Dropped {before - after} rows with missing values ({after} remain)")

print(data[target_col].value_counts())


Dropped 71 rows with missing values (1401 remain)
label
0    781
1    620
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

X = data[feature_cols]
y = data[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train size:", len(X_train), "Test size:", len(X_test))

Train size: 1120 Test size: 281


In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
model.fit(X_train, y_train)


RandomForestClassifier(max_depth=10, n_estimators=200, random_state=42)

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print()
print(classification_report(y_test, y_pred))
print()
print("Feature importance:")
for name, importance in sorted(
    zip(feature_cols, model.feature_importances_), key=lambda x: -x[1]
):
    print(f"  {name}: {importance:.3f}")

Accuracy: 0.9501779359430605
ROC-AUC: 0.9860797205670844

              precision    recall  f1-score   support

           0       0.94      0.97      0.96       157
           1       0.96      0.93      0.94       124

    accuracy                           0.95       281
   macro avg       0.95      0.95      0.95       281
weighted avg       0.95      0.95      0.95       281


Feature importance:
  dist_to_road: 0.541
  elevation: 0.160
  rainfall: 0.120
  slope: 0.111
  aspect: 0.069


In [ ]:
import joblib

joblib.dump(model, "landslide_model.pkl")
print("Saved landslide_model.pkl")

Saved landslide_model.pkl


In [ ]:
from google.colab import files
files.download("landslide_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
model.predict(X)

array([1, 1, 0, ..., 0, 0, 0])

In [ ]:
model.predict([[1250, 32, 180, 250, 1800]])

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


array([0])

In [2]:
!pip install rasterstats geopandas rasterio shapely joblib xarray netcdf4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 24.1 MB/s eta 0:00:00


In [3]:
import numpy as np
import pandas as pd
import geopandas as gpd
import joblib

from shapely.geometry import Point
from rasterstats import point_query

In [4]:
model = joblib.load("landslide_model.pkl")

print("Model loaded successfully")
print("Expected features:")
print(model.feature_names_in_)

Model loaded successfully
Expected features:
['elevation' 'slope' 'aspect' 'dist_to_road' 'rainfall']


In [5]:
import rasterio
import pandas as pd
import geopandas as gpd

# ==========================================
# 1. Inspect DEM
# ==========================================
with rasterio.open("DEM.tif") as src:
    print("===== DEM INFORMATION =====")
    print("CRS:", src.crs)
    print("Width:", src.width)
    print("Height:", src.height)
    print("Resolution:", src.res)
    print("Bounds:", src.bounds)
    print("NoData:", src.nodata)

# ==========================================
# 2. Inspect landslide CSV
# ==========================================
print("\n===== LANDSLIDE CSV =====")

landslide_df = pd.read_csv("landslide_points.csv")

print("Columns:")
print(landslide_df.columns.tolist())

print("\nFirst 5 rows:")
display(landslide_df.head())

print("\nShape:")
print(landslide_df.shape)

===== DEM INFORMATION =====
CRS: EPSG:4326
Width: 3431
Height: 3088
Resolution: (0.0002777777777778147, 0.0002777777777778147)
Bounds: BoundingBox(left=92.52041666670272, bottom=24.971249999995216, right=93.4734722222584, top=25.829027777773106)
NoData: -32768.0

===== LANDSLIDE CSV =====
Columns:
['geometry', 'label', 'elevation', 'slope', 'aspect', 'u_left', 'v_left', 'key_left', 'osmid_left', 'highway_left', 'oneway_left', 'reversed_left', 'length_left', 'bridge_left', 'ref_left', 'lanes_left', 'junction_left', 'name_left', 'tunnel_left', 'dist_to_road', 'rainfall', 'index_right', 'u_right', 'v_right', 'key_right', 'osmid_right', 'highway_right', 'oneway_right', 'reversed_right', 'length_right', 'bridge_right', 'ref_right', 'lanes_right', 'junction_right', 'name_right', 'tunnel_right']

First 5 rows:


,geometry,label,elevation,slope,aspect,u_left,v_left,key_left,osmid_left,highway_left,...,highway_right,oneway_right,reversed_right,length_right,bridge_right,ref_right,lanes_right,junction_right,name_right,tunnel_right
0,POINT (516548.80524672044 2765289.9437741293),1,210.068974,16.374961,261.680795,12205921413,6263645330,0,"[668863304, 667217706, 667217692, 525641655]",['unclassified'],...,['unclassified'],False,"[ false, true ]",5314.251054,yes,NH137G,NaN,NaN,NaN,NaN
1,POINT (516548.80524672044 2765289.9437741293),1,210.068974,16.374961,261.680795,6263645330,12205921413,0,"[668863304, 667217706, 667217692, 525641655]",['unclassified'],...,['unclassified'],False,"[ false, true ]",5314.251054,yes,NH137G,NaN,NaN,NaN,NaN
2,POINT (504036.12722508784 2765834.1721646464),1,NaN,NaN,NaN,12205988597,12205991116,0,1318888631,['residential'],...,['residential'],False,False,355.270201,NaN,NaN,NaN,NaN,NaN,NaN
3,POINT (504036.12722508784 2765834.1721646464),1,NaN,NaN,NaN,12205991116,12205988597,0,1318888631,['residential'],...,['residential'],False,False,355.270201,NaN,NaN,NaN,NaN,NaN,NaN
4,POINT (475260.98717746424 2766119.3788096285),1,NaN,NaN,NaN,9784622508,9784757148,0,1065443140,['residential'],...,['residential'],False,True,88.432842,NaN,NaN,NaN,NaN,NaN,NaN



Shape:
(1472, 36)


In [6]:
import rasterio
import pandas as pd

with rasterio.open("DEM.tif") as src:
    print("===== DEM INFORMATION =====")
    print("CRS:", src.crs)
    print("Width:", src.width)
    print("Height:", src.height)
    print("Resolution:", src.res)
    print("Bounds:", src.bounds)
    print("NoData:", src.nodata)

landslide_df = pd.read_csv("landslide_points.csv")

print("\n===== LANDSLIDE CSV =====")
print("Columns:", landslide_df.columns.tolist())
print("\nFirst 5 rows:")
display(landslide_df.head())
print("\nShape:", landslide_df.shape)

===== DEM INFORMATION =====
CRS: EPSG:4326
Width: 3431
Height: 3088
Resolution: (0.0002777777777778147, 0.0002777777777778147)
Bounds: BoundingBox(left=92.52041666670272, bottom=24.971249999995216, right=93.4734722222584, top=25.829027777773106)
NoData: -32768.0

===== LANDSLIDE CSV =====
Columns: ['geometry', 'label', 'elevation', 'slope', 'aspect', 'u_left', 'v_left', 'key_left', 'osmid_left', 'highway_left', 'oneway_left', 'reversed_left', 'length_left', 'bridge_left', 'ref_left', 'lanes_left', 'junction_left', 'name_left', 'tunnel_left', 'dist_to_road', 'rainfall', 'index_right', 'u_right', 'v_right', 'key_right', 'osmid_right', 'highway_right', 'oneway_right', 'reversed_right', 'length_right', 'bridge_right', 'ref_right', 'lanes_right', 'junction_right', 'name_right', 'tunnel_right']

First 5 rows:


,geometry,label,elevation,slope,aspect,u_left,v_left,key_left,osmid_left,highway_left,...,highway_right,oneway_right,reversed_right,length_right,bridge_right,ref_right,lanes_right,junction_right,name_right,tunnel_right
0,POINT (516548.80524672044 2765289.9437741293),1,210.068974,16.374961,261.680795,12205921413,6263645330,0,"[668863304, 667217706, 667217692, 525641655]",['unclassified'],...,['unclassified'],False,"[ false, true ]",5314.251054,yes,NH137G,NaN,NaN,NaN,NaN
1,POINT (516548.80524672044 2765289.9437741293),1,210.068974,16.374961,261.680795,6263645330,12205921413,0,"[668863304, 667217706, 667217692, 525641655]",['unclassified'],...,['unclassified'],False,"[ false, true ]",5314.251054,yes,NH137G,NaN,NaN,NaN,NaN
2,POINT (504036.12722508784 2765834.1721646464),1,NaN,NaN,NaN,12205988597,12205991116,0,1318888631,['residential'],...,['residential'],False,False,355.270201,NaN,NaN,NaN,NaN,NaN,NaN
3,POINT (504036.12722508784 2765834.1721646464),1,NaN,NaN,NaN,12205991116,12205988597,0,1318888631,['residential'],...,['residential'],False,False,355.270201,NaN,NaN,NaN,NaN,NaN,NaN
4,POINT (475260.98717746424 2766119.3788096285),1,NaN,NaN,NaN,9784622508,9784757148,0,1065443140,['residential'],...,['residential'],False,True,88.432842,NaN,NaN,NaN,NaN,NaN,NaN



Shape: (1472, 36)


In [7]:
# Install GDAL tools
!apt-get update -qq
!apt-get install -y -qq gdal-bin

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package python3-numpy.
(Reading database ... 118419 files and directories currently installed.)
Preparing to unpack .../python3-numpy_1%3a1.21.5-1ubuntu22.04.1_amd64.deb ...
Unpacking python3-numpy (1:1.21.5-1ubuntu22.04.1) ...
Selecting previously unselected package python3-gdal.
Preparing to unpack .../python3-gdal_3.8.4+dfsg-1~jammy0_amd64.deb ...
Unpacking python3-gdal (3.8.4+dfsg-1~jammy0) ...
Selecting previously unselected package gdal-bin.
Preparing to unpack .../gdal-bin_3.8.4+dfsg-1~jammy0_amd64.deb ...
Unpacking gdal-bin (3.8.4+dfsg-1~jammy0) ...
Setting up python3-numpy (1:1.21.5-1ubuntu22.04.1) ...
Setting up python3-gdal (3.8.4+dfsg-1~jammy0) ...
Setting up gdal-bin (3.8.4+dfsg-1~jammy0) ...
Processing triggers for man-db (2.10.2-1) ...


In [8]:
# Generate slope and aspect from DEM

!gdaldem slope DEM.tif slope.tif -s 111120
!gdaldem aspect DEM.tif aspect.tif

print("Created:")
print("✅ slope.tif")
print("✅ aspect.tif")

0...10...20...30...40...50...60...70...80...90...100 - done.
0...10...20...30...40...50...60...70...80...90...100 - done.
Created:
✅ slope.tif
✅ aspect.tif


In [9]:
import rasterio
import numpy as np

for filename in ["DEM.tif", "slope.tif", "aspect.tif"]:
    with rasterio.open(filename) as src:
        data = src.read(1)

        valid = data[data != src.nodata] if src.nodata is not None else data

        print("\n", filename)
        print("CRS:", src.crs)
        print("Shape:", data.shape)
        print("Min:", np.nanmin(valid))
        print("Max:", np.nanmax(valid))


 DEM.tif
CRS: EPSG:4326
Shape: (3088, 3431)
Min: 10
Max: 2069

 slope.tif
CRS: EPSG:4326
Shape: (3088, 3431)
Min: 0.0
Max: 78.17661

 aspect.tif
CRS: EPSG:4326
Shape: (3088, 3431)
Min: 0.0
Max: 359.83676


In [10]:
features = [
    "elevation",
    "slope",
    "aspect",
    "dist_to_road",
    "rainfall"
]

print("===== FIVE MODEL FEATURES =====")

print(landslide_df[features].describe())

print("\n===== MISSING VALUES =====")

print(landslide_df[features].isna().sum())

print("\n===== LABEL =====")

print(landslide_df["label"].value_counts())

===== FIVE MODEL FEATURES =====
         elevation        slope       aspect   dist_to_road     rainfall
count  1407.000000  1401.000000  1401.000000    1472.000000  1472.000000
mean    563.243274    16.177205   175.542125    2262.232523     5.384847
std     307.367763     8.146694    97.169583   11597.106895     1.799961
min      83.517195     0.314669     0.820042       0.194284     2.765881
25%     339.694114    10.067945    92.561407      51.084763     3.475105
50%     515.164199    15.714367   174.076943     460.576227     4.997047
75%     707.097164    21.169053   256.637244    2037.040299     6.778077
max    1716.988510    48.824283   357.209303  183870.345136     8.737246

===== MISSING VALUES =====
elevation       65
slope           71
aspect          71
dist_to_road     0
rainfall         0
dtype: int64

===== LABEL =====
label
0    785
1    687
Name: count, dtype: int64


In [11]:
features = [
    "elevation",
    "slope",
    "aspect",
    "dist_to_road",
    "rainfall"
]

print("===== FIVE MODEL FEATURES =====")
print(landslide_df[features].describe())

print("\n===== MISSING VALUES =====")
print(landslide_df[features].isna().sum())

print("\n===== LABEL =====")
print(landslide_df["label"].value_counts())

===== FIVE MODEL FEATURES =====
         elevation        slope       aspect   dist_to_road     rainfall
count  1407.000000  1401.000000  1401.000000    1472.000000  1472.000000
mean    563.243274    16.177205   175.542125    2262.232523     5.384847
std     307.367763     8.146694    97.169583   11597.106895     1.799961
min      83.517195     0.314669     0.820042       0.194284     2.765881
25%     339.694114    10.067945    92.561407      51.084763     3.475105
50%     515.164199    15.714367   174.076943     460.576227     4.997047
75%     707.097164    21.169053   256.637244    2037.040299     6.778077
max    1716.988510    48.824283   357.209303  183870.345136     8.737246

===== MISSING VALUES =====
elevation       65
slope           71
aspect          71
dist_to_road     0
rainfall         0
dtype: int64

===== LABEL =====
label
0    785
1    687
Name: count, dtype: int64


In [12]:
import os

print("DEM:", os.path.exists("DEM.tif"))
print("Slope:", os.path.exists("slope.tif"))
print("Aspect:", os.path.exists("aspect.tif"))

DEM: True
Slope: True
Aspect: True


In [14]:
import os

print("DEM:", os.path.exists("DEM.tif"))
print("Slope:", os.path.exists("slope.tif"))
print("Aspect:", os.path.exists("aspect.tif"))
print("OSM:", os.path.exists("north-eastern-zone-260904.osm.pbf"))

DEM: True
Slope: True
Aspect: True
OSM: True


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [15]:
from google.colab import files

uploaded = files.upload()

Saving RF25_indselect_rfp25.nc to RF25_indselect_rfp25.nc


In [21]:
import os
import glob

print("===== /content files =====")

for f in glob.glob("/content/*"):
    print(os.path.basename(f))

===== /content files =====
RF25_indselect_rfp25.nc
landslide_points.csv
DEM.tif
slope.tif
north-eastern-zone-260904.osm.pbf
landslide_model.pkl
RF25_ind2025_rfp25.nc
aspect.tif
sample_data


In [26]:
import os

f = "/content/RF25_ind2025_rfp25.nc"

print("Exists:", os.path.exists(f))

if os.path.exists(f):
    print("Size:", round(os.path.getsize(f) / (1024**2), 2), "MB")

Exists: True
Size: 24.25 MB


In [28]:
import xarray as xr

rainfall_file = "/content/RF25_ind2025_rfp25.nc"

rain_ds = xr.open_dataset(rainfall_file)

print(rain_ds)

<xarray.Dataset> Size: 25MB
Dimensions:    (TIME: 365, LATITUDE: 129, LONGITUDE: 135)
Coordinates:
  * TIME       (TIME) datetime64[ns] 3kB 2025-01-01 2025-01-02 ... 2025-12-31
  * LATITUDE   (LATITUDE) float64 1kB 6.5 6.75 7.0 7.25 ... 38.0 38.25 38.5
  * LONGITUDE  (LONGITUDE) float64 1kB 66.5 66.75 67.0 ... 99.5 99.75 100.0
Data variables:
    RAINFALL   (TIME, LATITUDE, LONGITUDE) float32 25MB ...
Attributes:
    history:      FERRET V6.82    9-Apr-26
    Conventions:  CF-1.0


In [29]:
import xarray as xr

rainfall_file = "/content/RF25_ind2025_rfp25.nc"

rain_ds = xr.open_dataset(rainfall_file)

print("===== RAINFALL DATASET =====")
print(rain_ds)

print("\n===== VARIABLES =====")
print(list(rain_ds.data_vars))

print("\n===== COORDINATES =====")
print(list(rain_ds.coords))

===== RAINFALL DATASET =====
<xarray.Dataset> Size: 25MB
Dimensions:    (TIME: 365, LATITUDE: 129, LONGITUDE: 135)
Coordinates:
  * TIME       (TIME) datetime64[ns] 3kB 2025-01-01 2025-01-02 ... 2025-12-31
  * LATITUDE   (LATITUDE) float64 1kB 6.5 6.75 7.0 7.25 ... 38.0 38.25 38.5
  * LONGITUDE  (LONGITUDE) float64 1kB 66.5 66.75 67.0 ... 99.5 99.75 100.0
Data variables:
    RAINFALL   (TIME, LATITUDE, LONGITUDE) float32 25MB ...
Attributes:
    history:      FERRET V6.82    9-Apr-26
    Conventions:  CF-1.0

===== VARIABLES =====
['RAINFALL']

===== COORDINATES =====
['LONGITUDE', 'LATITUDE', 'TIME']


In [30]:
print("===== VARIABLE DETAILS =====")

for var in rain_ds.data_vars:
    print("\nVariable:", var)
    print("Dimensions:", rain_ds[var].dims)
    print("Shape:", rain_ds[var].shape)
    print("Units:", rain_ds[var].attrs.get("units"))
    print("Long name:", rain_ds[var].attrs.get("long_name"))
    print("Minimum:", float(rain_ds[var].min()))
    print("Maximum:", float(rain_ds[var].max()))

===== VARIABLE DETAILS =====

Variable: RAINFALL
Dimensions: ('TIME', 'LATITUDE', 'LONGITUDE')
Shape: (365, 129, 135)
Units: mm
Long name: Rainfall
Minimum: 0.0
Maximum: 469.21014404296875


In [31]:
print("===== COORDINATES =====")

for coord in rain_ds.coords:
    values = rain_ds[coord].values

    print("\n", coord)
    print("Shape:", values.shape)
    print("First values:", values.flatten()[:5])
    print("Min:", float(values.min()))
    print("Max:", float(values.max()))

===== COORDINATES =====

 LONGITUDE
Shape: (135,)
First values: [66.5  66.75 67.   67.25 67.5 ]
Min: 66.5
Max: 100.0

 LATITUDE
Shape: (129,)
First values: [6.5  6.75 7.   7.25 7.5 ]
Min: 6.5
Max: 38.5

 TIME
Shape: (365,)
First values: ['2025-01-01T00:00:00.000000000' '2025-01-02T00:00:00.000000000'
 '2025-01-03T00:00:00.000000000' '2025-01-04T00:00:00.000000000'
 '2025-01-05T00:00:00.000000000']
Min: 1.7356896e+18
Max: 1.7671392e+18
